# Project 2 · Modules 1–3: products, clusters, recommender

One notebook for everything that runs on the final Phase 1 KG (reviewed copy, 13 Oct; read-only). No extraction, no LLM. Run top to bottom (about 30 s). Each section writes to its own folder under `data/processed/`. Market data is separate: `run_finance.ipynb` (network, rate-limited, run once a day at most).

In [ ]:
# Setup: run from a clone of the repo, locally or in Colab
import os, sys, json
from pathlib import Path

REPO_URL = "https://github.com/roolau/semiconductor-supply-chain-kg-extension.git"   # Colab only: the repo to clone
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("/content/repo").exists():
    !git clone -q {REPO_URL} /content/repo
ROOT = Path("/content/repo") if IN_COLAB else Path.cwd().resolve().parents[1]   # notebooks/ -> extension -> repo
EXT = ROOT / "equity-research-extension"   # this extension: code, reference data, outputs

# The Phase 1 KG (read-only): the folder holding fused_edges.jsonl and canonical_entities.jsonl
KG = Path(os.environ.get("P1_KG_DIR", ROOT / "data"))   # data/ at the repo root
assert (EXT / "src/kg/load.py").exists(), f"extension code not found under {EXT}"
assert (KG / "fused_edges.jsonl").exists(), "Phase 1 KG not found: set P1_KG_DIR to the folder with fused_edges.jsonl"
os.environ["P1_KG_DIR"] = str(KG)
if str(EXT) not in sys.path:
    sys.path.insert(0, str(EXT))

In [ ]:
import importlib; importlib.invalidate_caches()  
import pandas as pd
from src.kg import categories, load, products, eto, cluster, viz, recommend
for m in (categories, load, products, eto, cluster, viz, recommend):
    importlib.reload(m)
kg = load.load()
print(len(kg.edges), 'edges,', len(kg.supply_edges()), 'supply edges')

## 1 · Product-mediated relations
Reads the final Phase 1 KG (reviewed copy, 13 Oct) read-only and writes item categories, company product profiles and the gate coverage report to `data/processed/products/`.

In [ ]:
cov = products.build(kg, EXT / 'data/processed/products')
print(json.dumps({k: v for k, v in cov.items() if k != 'items_unmapped'}, indent=1))
print(len(cov['items_unmapped']), 'unmapped items:', cov['items_unmapped'])

Expected (16 Nov, strict=True: an item edge counts only with an SEC source, 2+ claims or HIGH confidence): 2,988 edges, 348 supply edges; edges explained leave-one-out 9% (exact items) → 16% (categories) → 46% (+ what customers buy from their other suppliers); gate passes 12.1% of company pairs; where both profiles exist it holds for 161 of 161 edges. `products.build(kg, ..., strict=False)` gives the unfiltered numbers (48%, 13.4%).

## 2 · Clustering (product, ecosystem, role)
Final Phase 1 KG, read-only. No extraction, no LLM. Writes `clusters.jsonl` and `clustering_report.json` to `data/processed/clusters/`. Role evaluation uses the CSET ETO data in `data/reference/eto/` (CC BY-NC 4.0).

In [ ]:
PARAMS = dict(threshold=0.5, resolution=1.0, k=8)   # product cut distance, Louvain resolution, unsupervised k
rep_c = cluster.build(kg, EXT / 'data/processed/clusters', **PARAMS)
print(json.dumps({k: rep_c[k] for k in ('product', 'ecosystem', 'role')}, indent=1))
print(json.dumps(rep_c['evaluation'], indent=1))

In [ ]:
import pandas as pd
print('Same product cluster, different ecosystem (insight-slide candidates):')
display(pd.DataFrame(rep_c['disagreements'])[['a', 'b', 'ecosystem_a', 'ecosystem_b', 'makes_shared']])
df = pd.read_json(EXT / 'data/processed/clusters/clusters.jsonl', lines=True)
display(df[df.eto_role.notna()][['name', 'role', 'tier', 'tier_from', 'eto_role', 'eto_stage']].sort_values('eto_role'))

In [ ]:
# Interactive map: colour by ecosystem / role / tier / ETO grading, click a company for its partners
from IPython.display import HTML
page = viz.write(kg, EXT / "data/processed/clusters/clusters_map.html")
HTML(page.read_text())

Expected (16 Nov, same files): product 67 companies / 32 clusters; ecosystem 238 companies / 14 communities, modularity ≈ 0.60; ETO-matched 38. Role from products 19/20 correct (95%); tier from products 19/20, from customers 6/6, from suppliers 0/7 (buyer-only companies such as Tesla and Google are graded by what they sell in ETO); unsupervised Ward 9/33 (comparison only). Stage: from products 19/20, Phase 1 LLM stage labels 11/13 — higher precision and more coverage (50% vs 29% of the 38).

## 3 · Relationship recommender
Who could become this company's next customer (or supplier), with the evidence path behind each suggestion. Final Phase 1 KG, read-only; no extraction. Evaluation: hide 20% of supply links (every company keeps at least one), rebuild everything on the rest, and rank each hidden partner among all 238 companies. Hits@10 = hidden partner in the top 10; MRR = mean of 1/rank.

In [ ]:
rep = recommend.build(kg, EXT / "data/processed/recommender")   # about 10 s
for d in ("customers", "suppliers"):
    print(f"\nRecommend {d}: {rep[d]['test_edges_per_seed'][0]} hidden links x {len(rep[d]['seeds'])} seeds")
    display(pd.DataFrame(rep[d]["methods"]).T)
display(pd.DataFrame(rep["customers"]["hybrid_gate_hits10_by_stage"]).T.rename_axis("supplier stage"))

In [ ]:
# Try any company: recommendations on the full graph, with evidence paths
COMPANY, DIRECTION, K = "ASML", "customers", 5     # DIRECTION: "customers" or "suppliers"
cid = next(e for e, v in kg.entities.items() if v.get("entity_type") == "Company" and v["canonical_name"].lower() == COMPANY.lower())
R = recommend.Recommender(kg, kg.supply_edges(), direction=DIRECTION)
print(f"{COMPANY} today {'supplies' if DIRECTION == 'customers' else 'buys from'}:", ", ".join(sorted(kg.name(x) for x in R.out[cid])))
for r in R.recommend(cid, K):
    print(f"\n{r['name']}  (score {r['score']})")
    for e in r["evidence"]:
        print("   -", e)

Expected (16 Nov, 5 seeds, 69 hidden links each): recommend customers: hybrid + gate MRR 0.269, Hits@10 0.545 vs popularity 0.154 / 0.441; recommend suppliers: hybrid + gate 0.315 / 0.594 vs popularity 0.211 / 0.487. Ablations: without peers Hits@10 drops most, without products MRR drops most; adding the stage or ecosystem scorer does not help (they stay in the explanations and the map).

## 4 · Build the website
Patches the extension modules into the Phase 1 page (`site/phase1_base.html`) and writes one self-contained HTML file. Uses the saved results in `data/results/` (market-data snapshot and the KGQA held-out run), so no network or API is needed.

In [ ]:
from src.kg import site
importlib.reload(site)
R = EXT / 'data/results'
page = site.build(EXT / 'site/phase1_base.html', kg, EXT / 'site/index.html',
                  finance_snapshot=R / 'finance/yfinance_snapshot.json',
                  qa_results=R / 'qa/seed11/results_llm-v2_paraphrase.jsonl',
                  qa_report=R / 'qa/seed11/qa_report.json')
print(page, f'{page.stat().st_size / 1e6:.1f} MB')